# Spotify 550K Preprocessing

In [ ]:
import pandas as pd
import ast # for converting strings to Python lists

# connect to Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Load data

If needed, add a shortcut for the [CIS5500 Group Project folder](https://drive.google.com/drive/folders/1sepjOtyL6yT71mu2yo0aV1uCv0SVcMTg?usp=sharing) to My Drive
- Right click CIS5500 Group Project > Organize > Add shortcut > All locations > My Drive

In [ ]:
SONGS_PATH = "/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550/songs.csv"
ARTISTS_PATH = "/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550/artists.csv"

In [ ]:
# read songs data and rename columns
songs = pd.read_csv(
    SONGS_PATH
).rename(
    {
        "id": "song_id",
        "name": "song_name"
    },
    axis=1
)

songs.head()

,song_id,song_name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,...,tempo,duration_ms,lyrics,year,genre,popularity,total_artist_followers,avg_artist_popularity,artist_ids,niche_genres
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,"[""HELLYEAH""]",0.415,0.605,7,-11.157,1,0.0575,...,100.059,79500,"He said he came from Jamaica,\nhe owned a coup...",2016,Rock,0,769490,52.0,"[""4hxDvVq5t8ebPYPdBl1F9f""]","[""groove metal"", ""metal""]"
1,2ASl4wirkeYm3OWZxXKYuq,!!,Childhood Dreams,"[""Yxngxr1""]",0.788,0.648,7,-9.135,0,0.3150,...,79.998,114000,"Fuck the bitch, now she running with my kids\n...",2019,Hip-Hop,29,143628,45.0,"[""2jwRHcdgkRhelYEMqndDKe""]",[]
2,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,Situationist Comedy,"[""Dillinger Four""]",0.171,0.957,2,-5.749,1,0.1490,...,175.317,197400,You like to stand on the other side\nPoint and...,2002,Rock,0,36619,35.0,"[""4YAN46l70QV0PGXlMg0iHi""]","[""melodic hardcore"", ""pop punk"", ""punk"", ""skat..."
3,0fROT4kK5oTm8xO8PX6EJF,!I'll Be Back!,!I'll Be Back!,"[""Ril\u00e8s""]",0.823,0.612,1,-7.767,1,0.2480,...,142.959,178533,"It's been a while, shit, I missed the rehab, p...",2018,Hip-Hop,43,929303,63.0,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"
4,1xBFhv5faebv3mmwxx7DnS,!Lost!,!Lost!,"[""Ril\u00e8s""]",0.729,0.552,7,-8.562,0,0.0650,...,86.103,186197,I would like to give you all my time\nI would ...,2018,Hip-Hop,0,929303,63.0,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"


In [ ]:
# read artists data and rename columns
artists = pd.read_csv(
    ARTISTS_PATH
).rename(
    {
        "id": "artist_id",
        "name": "artist_name"
    },
    axis=1
)

artists.head()

,artist_id,artist_name,followers,popularity,genres,main_genre
0,6YROFUbu5zRCHi2xkir5pk,Brian Hyland,67223,47,[],Pop
1,5tFRohaO5yEsuJxmMnlCO9,Barns Courtney,602647,62,[],Electronic
2,3w1Q754jb31h5CXQCcnLNL,Capcom Sound Team,210392,58,"['japanese vgm', 'soundtrack']",Electronic
3,3oDbviiivRWhXwIE8hxkVV,The Beach Boys,5139194,76,['baroque pop'],Classical
4,60zvRmhQHRxokEB1taAVpN,Beth Malone,1569,29,['musicals'],Classical


## Normalization

### Albums Info

In [ ]:
# subset columns
albums_info = songs.copy()[["song_id", "artist_ids", "album_name"]]

# unnest artist ids into separate rows
albums_info["artist_id"] = albums_info["artist_ids"].apply(ast.literal_eval)
albums_info = (
    albums_info
    .explode("artist_id")
    .drop("artist_ids", axis=1)
)

# for each song, get the first artist id (assumed to be the main artist)
albums_info = (
    albums_info
    .groupby(["song_id", "album_name"], as_index=False)["artist_id"]
    .first()
)

# for each album name and artist id pair, give it a unique index (+ 1 to make the index start from one)
albums_info["album_id"] = (
    albums_info[["album_name", "artist_id"]]
    .groupby(["album_name", "artist_id"])
    .ngroup()
) + 1

# clean up output
albums_info = (
    albums_info[["album_id", "album_name", "artist_id"]]
    .drop_duplicates()
    .sort_values("album_id")
    .reset_index(drop=True)
)

albums_info

,album_id,album_name,artist_id
0,1,!,6Xgp2XMz1fhVYe7i6yNAax
1,2,!!!Going Places!!!,09L3cUdx0hq6qn5bKuJJ4I
2,3,!I'll Be Back!,6pdcQa7by8IKuoVXvgknlI
3,4,!K7 Kollections 02: Classics,1hwAhXzyuEUjug2pyNVSvg
4,5,!Lost!,6pdcQa7by8IKuoVXvgknlI
...,...,...,...
184759,184760,８６―エイティシックス― オリジナル・サウンドトラック,0Riv2KnFcLZA3JSVryRg4y
184760,184761,ＡＫＩＮＡ ＢＯＸ,7140bcJ0ZySe314nUfOo1J
184761,184762,ＴＯＭＭＹ ＩＣＥ ＣＲＥＡＭ ＨＥＡＶＥＮ ＦＯＲＥＶＥＲ,6ClPIi6VMHv2Q3OZ4R17wV
184762,184763,［Dystopia : Lose Myself］,5V1qsQHdXNm4ZEZHWvFnqQ


**Note:** There are 20 songs which do not have an associated album.

In [ ]:
songs[songs["album_name"].isna()].shape[0]

20

### Songs Info (Metadata)

In [ ]:
# subset columns
spotify_songs = songs.copy()[["song_id", "song_name", "album_name", "year", "artist_ids"]]

# unnest artist ids into separate rows
spotify_songs["artist_id"] = spotify_songs["artist_ids"].apply(ast.literal_eval)
spotify_songs = (
    spotify_songs
    .explode("artist_id")
    .drop("artist_ids", axis=1)
)

# for each song, get the first artist id (assumed to be the main artist)
spotify_songs = (
    spotify_songs
    .groupby(["song_id", "song_name", "album_name", "year"], as_index=False, dropna=False)["artist_id"]
    .first()
)

# merge songs and albums info in order to get album id
spotify_songs = spotify_songs.merge(
    albums_info,
    how="left",
    on=["album_name", "artist_id"]
)

# convert album id column to nullable integer type
spotify_songs["album_id"] = spotify_songs["album_id"].astype('Int64')

# subset columns in output
spotify_songs = spotify_songs[["song_id", "song_name", "album_id", "year"]]

spotify_songs

,song_id,song_name,album_id,year
0,0001Lyv0YTjkZSqzT4WkLy,Eye Of The Hurricane,53745,1993
1,0001piYJu94Ec4hJFytG5G,Nothing but a Shade,154648,2017
2,0005VnpISGYLSGrXg9TEJS,Space Loneliness,7761,2009
3,000B6fUCVkSThxEbewDZ8r,Fight or Flight,98021,2020
4,000CC8EParg64OmTxVnZ0p,It's All Coming Back To Me Now - Cover of Céli...,60043,2021
...,...,...,...,...
550617,7zzaZg8MollFiBDSbuylvZ,Take The Fifth,59714,2001
550618,7zzbfi8fvHe6hm342GcNYl,Black-Throated Wind,8091,1972
550619,7zzcBFlsapZWEyAxT3N1II,Iceberg,142451,2007
550620,7zzncKm7SiyPzmQGbWR3VO,This Missin' You Heart Of Mine,136863,1987


**Sanity check:** There should be 20 missing album IDs.

In [ ]:
spotify_songs[spotify_songs["album_id"].isna()].shape

(20, 4)

### Songs Subgenres

In [ ]:
# subset columns
songs_subgenres = songs.copy()[["song_id", "niche_genres"]]

# unnest values into separate rows
songs_subgenres["niche_genres"] = songs_subgenres["niche_genres"].apply(ast.literal_eval)
songs_subgenres = songs_subgenres.explode("niche_genres")

# rename column
songs_subgenres = songs_subgenres.rename(
    {"niche_genres": "subgenre"},
    axis=1
)

# reset index
songs_subgenres = songs_subgenres.reset_index(drop=True)

songs_subgenres

,song_id,subgenre
0,0Prct5TDjAnEgIqbxcldY9,groove metal
1,0Prct5TDjAnEgIqbxcldY9,metal
2,2ASl4wirkeYm3OWZxXKYuq,NaN
3,5tA3ImW310llKo8EMBj2Ga,melodic hardcore
4,5tA3ImW310llKo8EMBj2Ga,pop punk
...,...,...
1597845,7ItLwpnDJNmnmjlx4hOC96,black metal
1597846,773j5fNWIjO0EWqAiX8Quo,black metal
1597847,0fMJpTECbr7MwQYJjopAWf,big beat
1597848,6yL5qOKgvBgn4H0XJEDyAV,christmas


### Audio Attributes

In [ ]:
# mappings for key and mode (minor / major)
KEY_MAP = {
    0: "C", 1: "C#", 2: "D", 3: "D#",
    4: "E", 5: "F", 6: "F#", 7: "G",
    8: "G#", 9: "A", 10: "A#", 11: "B"
}

MODE_MAP = {
    0: "minor",
    1: "major"
}

# create columns for key (e.g., G major) and duration (seconds)
audio_attributes = songs.copy()
audio_attributes["key"] = audio_attributes["key"].map(KEY_MAP) + " " + audio_attributes["mode"].map(MODE_MAP)
audio_attributes["duration_sec"] = audio_attributes["duration_ms"] / 1000

# subset columns
audio_attributes = audio_attributes[[
    "song_id", "tempo", "popularity", "key", "genre", "duration_sec", "danceability", "energy", "loudness",
    "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "popularity"
]]

# add embeddings
audio_attributes["embedding"] = audio_attributes[[
    "danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "popularity"
]].values.tolist()

audio_attributes

,song_id,tempo,popularity,key,genre,duration_sec,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,popularity,embedding
0,0Prct5TDjAnEgIqbxcldY9,100.059,0,G major,Rock,79.500,0.4150,0.605,-11.157,0.0575,0.001160,0.838000,0.4710,0.1930,0,"[0.415, 0.605, -11.157, 0.0575, 0.00116, 0.838..."
1,2ASl4wirkeYm3OWZxXKYuq,79.998,29,G minor,Hip-Hop,114.000,0.7880,0.648,-9.135,0.3150,0.900000,0.000000,0.1760,0.2870,29,"[0.788, 0.648, -9.135, 0.315, 0.9, 0.0, 0.176,..."
2,5tA3ImW310llKo8EMBj2Ga,175.317,0,D major,Rock,197.400,0.1710,0.957,-5.749,0.1490,0.000029,0.000032,0.3300,0.3490,0,"[0.171, 0.957, -5.749, 0.149, 2.9e-05, 3.20000..."
3,0fROT4kK5oTm8xO8PX6EJF,142.959,43,C# major,Hip-Hop,178.533,0.8230,0.612,-7.767,0.2480,0.168000,0.000000,0.1090,0.6880,43,"[0.8230000000000001, 0.612, -7.767, 0.248, 0.1..."
4,1xBFhv5faebv3mmwxx7DnS,86.103,0,G minor,Hip-Hop,186.197,0.7290,0.552,-8.562,0.0650,0.183000,0.000000,0.1310,0.3800,0,"[0.729, 0.552, -8.562, 0.065, 0.183, 0.0, 0.13..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550617,7ItLwpnDJNmnmjlx4hOC96,140.627,30,G major,Rock,477.294,0.2370,0.335,-11.895,0.0499,0.581000,0.740000,0.1530,0.0587,30,"[0.237, 0.335, -11.895, 0.0499, 0.581, 0.74, 0..."
550618,773j5fNWIjO0EWqAiX8Quo,85.559,38,G minor,Rock,356.752,0.0966,0.356,-8.289,0.0418,0.004800,0.509000,0.1120,0.0562,38,"[0.0966, 0.356, -8.289, 0.0418, 0.0048, 0.509,..."
550619,0fMJpTECbr7MwQYJjopAWf,134.967,3,B minor,Electronic,191.922,0.4870,0.698,-12.785,0.0455,0.000049,0.631000,0.3720,0.5440,3,"[0.487, 0.6980000000000001, -12.785, 0.0455, 4..."
550620,6yL5qOKgvBgn4H0XJEDyAV,120.035,24,D# major,Pop,159.267,0.7390,0.737,-5.372,0.0316,0.001610,0.000000,0.0458,0.7860,24,"[0.739, 0.737, -5.372, 0.0316, 0.00161, 0.0, 0..."


### Featured Artists

In [ ]:
# subset columns
featured_in = songs.copy()[["song_id", "artist_ids"]]

# unnest values into separate rows
featured_in["artist_ids"] = featured_in["artist_ids"].apply(ast.literal_eval)
featured_in = featured_in.explode("artist_ids")

# rename columns
featured_in = featured_in.rename({
    "artist_ids": "artist_id"
}, axis=1)

# reset index
featured_in = featured_in.reset_index(drop=True)

featured_in

,song_id,artist_id
0,0Prct5TDjAnEgIqbxcldY9,4hxDvVq5t8ebPYPdBl1F9f
1,2ASl4wirkeYm3OWZxXKYuq,2jwRHcdgkRhelYEMqndDKe
2,5tA3ImW310llKo8EMBj2Ga,4YAN46l70QV0PGXlMg0iHi
3,0fROT4kK5oTm8xO8PX6EJF,6pdcQa7by8IKuoVXvgknlI
4,1xBFhv5faebv3mmwxx7DnS,6pdcQa7by8IKuoVXvgknlI
...,...,...
679264,0fMJpTECbr7MwQYJjopAWf,4e0ETTbxMHwq7DtH9LSHAJ
679265,6yL5qOKgvBgn4H0XJEDyAV,4uN3DsfENc7dp0OLO0FEIb
679266,6yL5qOKgvBgn4H0XJEDyAV,4V8U8U6LwsHGyRTLCt9t19
679267,036JzAN5DCANSZyeW6MjqG,4ylR3zwA0zaapAu94fktwa


### Artists Info

In [ ]:
# subset columns
artists_info = artists[["artist_id", "artist_name", "followers", "popularity", "main_genre"]]

# rename column
artists_info = artists_info.rename({
    "main_genre": "genre"
}, axis=1)

artists_info

,artist_id,artist_name,followers,popularity,genre
0,6YROFUbu5zRCHi2xkir5pk,Brian Hyland,67223,47,Pop
1,5tFRohaO5yEsuJxmMnlCO9,Barns Courtney,602647,62,Electronic
2,3w1Q754jb31h5CXQCcnLNL,Capcom Sound Team,210392,58,Electronic
3,3oDbviiivRWhXwIE8hxkVV,The Beach Boys,5139194,76,Classical
4,60zvRmhQHRxokEB1taAVpN,Beth Malone,1569,29,Classical
...,...,...,...,...,...
71435,7zO9lJlwwLENgvHlNibFl4,A P,89,0,Rock
71436,7zgrApXwTCzakPbbBSQa74,K.Tee,367,12,Electronic
71437,7zpI3b0T03LRvp6UqyjS2g,RB Stone,434,6,Blues
71438,7zvAEPfm6K6WSg67OkVcIu,Sabotage Soundsystem,3811,13,Folk


### Artists Subgenres

In [ ]:
# subset columns
artists_subgenres = artists.copy()[["artist_id", "genres"]]

# unnest values into separate rows
artists_subgenres["genres"] = artists_subgenres["genres"].apply(ast.literal_eval)
artists_subgenres = artists_subgenres.explode("genres")

# rename column
artists_subgenres = artists_subgenres.rename({
    "genres": "subgenre"
}, axis=1)

# reset index
artists_subgenres = artists_subgenres.reset_index(drop=True)

artists_subgenres

,artist_id,subgenre
0,6YROFUbu5zRCHi2xkir5pk,NaN
1,5tFRohaO5yEsuJxmMnlCO9,NaN
2,3w1Q754jb31h5CXQCcnLNL,japanese vgm
3,3w1Q754jb31h5CXQCcnLNL,soundtrack
4,3oDbviiivRWhXwIE8hxkVV,baroque pop
...,...,...
135097,7zpI3b0T03LRvp6UqyjS2g,blues rock
135098,7zpI3b0T03LRvp6UqyjS2g,blues
135099,7zvAEPfm6K6WSg67OkVcIu,reggae rock
135100,7zvAEPfm6K6WSg67OkVcIu,reggae


## Save data

In [ ]:
# # save data in Google Drive
spotify_songs.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550_cleaned/spotify_songs.csv", index=False)
songs_subgenres.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550_cleaned/songs_subgenres.csv", index=False)
audio_attributes.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550_cleaned/audio_attributes.csv", index=False)
featured_in.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550_cleaned/featured_in.csv", index=False)
albums_info.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550_cleaned/albums_info.csv", index=False)
artists_info.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550_cleaned/artists_info.csv", index=False)
artists_subgenres.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550_cleaned/artists_subgenres.csv", index=False)